<a href="https://colab.research.google.com/github/Ted-star7/Rev1/blob/main/Rev1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uzapoint Liquor POS – Category Architecture Extraction
### Source: Uzapoint_Master_Clean_Data.csv
### Objective:
- Extract parent categories
- Extract subcategories
- Analyze distribution
- Design controlled Liquor POS taxonomy

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as numpy

#load the data
file_path = '/content/drive/MyDrive/Dataset/Uzapoint_Master_Clean_Data.csv'
df = pd.read_csv(file_path)

#display the first 5 rows
print("Successfully loaded data. Here are the first 5 rows:")
print(df.head())

Successfully loaded data. Here are the first 5 rows:
   Product ID             Product Label       Clean_Product_Label  \
0        8782                Brown Belt                Brown Belt   
1        8783                Black Belt                Black Belt   
2        8785          Apple Watch 2020          Apple Watch 2020   
3        8797          Apple Watch 2019          Apple Watch 2019   
4      115430  ORAIMO TYPE C DATA CABLE  ORAIMO TYPE C DATA CABLE   

      Category     Subcategory Parent_Category Category_norm Subcategory_norm  \
0  ACCESSORIES           BELTS     Still_Other   accessories            belts   
1  ACCESSORIES           BELTS     Still_Other   accessories            belts   
2  ACCESSORIES         WATCHES     Still_Other   accessories          watches   
3  ACCESSORIES         WATCHES     Still_Other   accessories          watches   
4  ACCESSORIES  PHONE CHARGERS     Still_Other   accessories   phone chargers   

    Volume   Brand                           

In [3]:
df['Parent_Category'].value_counts().head(20)


,count
Parent_Category,
spirits,21473
wines,8502
non-alcoholic,5401
beers,3984
household_kitchen,2749
snacks_misc,2277
Still_Other,1108
tobacco,1023
miscellaneous_retail,731


In [4]:
df['Category'].value_counts().head(30)

,count
Category,
WHISKY,4654
WINE,4208
VODKA,2872
BEERS,2567
GIN,2261
EXTRAS,2246
WINES,2178
PRODUCTS,1740
WHISKEY,1735


In [5]:
df['Subcategory'].value_counts().head(30)

,count
Subcategory,
ALL VODKA,2475
ALL GIN,1832
BLENDED SCOTCH,1530
SOFT DRINKS,1256
ALL WHISKY,1211
CANS,1158
RED WINE,1077
ALL WHISKEY,1063
RED,1019


# Uzapoint Liquor POS – Parent Category Engineering

## Objective
Redesign the Parent Category structure of the Uzapoint POS system
using the master dataset (47,944 rows) to create a controlled,
scalable Liquor POS taxonomy.

---

## Problem Statement

The current system contains 17+ Parent Categories including:

- household_kitchen
- snacks_misc
- miscellaneous_retail
- promotional_items
- grocery_food
- operational_expenses
- etc.

This creates:
- Category explosion
- Reporting fragmentation
- Onboarding friction
- Inconsistent hierarchy

---

## Goal

Compress all Parent Categories into a clean, controlled POS structure:

1. Spirits
2. Wines
3. Beers
4. Non-Alcoholic
5. Tobacco
6. Others

This ensures:
- Simplicity
- Liquor-first design
- Backward compatibility
- Structured reporting

In [6]:
## Step 1: Normalize Raw Parent Categories

#We standardize text formatting to prevent case-based duplication
#(e.g., "Spirits" vs "spirits").

df['Parent_Category'] = df['Parent_Category'].str.strip().str.lower()

df['Parent_Category'].value_counts().head(15)

,count
Parent_Category,
spirits,21473
wines,8502
non-alcoholic,5401
beers,3984
household_kitchen,2749
snacks_misc,2277
still_other,1108
tobacco,1023
miscellaneous_retail,731


In [8]:
## Step 2: Define Controlled POS Parent Mapping
## We map all existing Parent Categories into 6 governed POS Parents.
## All non-liquor retail spillovers are absorbed into "Others".

def map_parent_category(raw_parent):

    if raw_parent == 'spirits':
        return 'Spirits'

    elif raw_parent == 'wines':
        return 'Wines'

    elif raw_parent == 'beers':
        return 'Beers'

    elif raw_parent in ['non-alcoholic', 'energy_functional_drinks']:
        return 'Non-Alcoholic'

    elif raw_parent == 'tobacco':
        return 'Tobacco'

    else:
        return 'Others'

df['POS_Parent_Category'] = df['Parent_Category'].apply(map_parent_category)

df['POS_Parent_Category'].value_counts()



,count
POS_Parent_Category,
Spirits,21473
Wines,8502
Others,7556
Non-Alcoholic,5406
Beers,3984
Tobacco,1023


In [9]:
## Step 3: Validate New Parent Distribution
#We analyze the percentage distribution of the new POS structure
#to ensure the taxonomy reflects actual business volume.

(df['POS_Parent_Category'].value_counts(normalize=True) * 100).round(2)

,proportion
POS_Parent_Category,
Spirits,44.79
Wines,17.73
Others,15.76
Non-Alcoholic,11.28
Beers,8.31
Tobacco,2.13


## Architectural Impact

This restructuring:

- Reduces Parent Categories from 17+ to 6
- Preserves all historical data
- Improves reporting clarity
- Simplifies onboarding for new liquor stores
- Prevents future category explosion
- Separates promotions from inventory hierarchy

This is a governance-layer redesign, not a data deletion.

# Export – POS Parent Categories

We export the finalized POS Parent structure into CSV files
for documentation, system integration, and migration planning.

Files to generate:
1. pos_parent_categories.csv
2. pos_parent_distribution.csv
3. parent_mapping_reference.csv

In [10]:
# Extract unique POS parent categories
pos_parents = pd.DataFrame(df['POS_Parent_Category'].unique(), columns=['POS_Parent_Category'])

# Sort alphabetically
pos_parents = pos_parents.sort_values(by='POS_Parent_Category')

# Save to CSV
pos_parents.to_csv("pos_parent_categories.csv", index=False)

pos_parents

,POS_Parent_Category
1,Beers
2,Non-Alcoholic
0,Others
3,Spirits
4,Tobacco
5,Wines


In [11]:
# Create distribution table
parent_distribution = df['POS_Parent_Category'].value_counts().reset_index()
parent_distribution.columns = ['POS_Parent_Category', 'Count']

# Add percentage column
parent_distribution['Percentage'] = (
    parent_distribution['Count'] / parent_distribution['Count'].sum() * 100
).round(2)

# Save
parent_distribution.to_csv("pos_parent_distribution.csv", index=False)

parent_distribution

#POS Mapping Reference
mapping_reference = df[['Parent_Category', 'POS_Parent_Category']].drop_duplicates()

mapping_reference = mapping_reference.sort_values(by='Parent_Category')

mapping_reference.to_csv("parent_mapping_reference.csv", index=False)

mapping_reference.head(20)

,Parent_Category,POS_Parent_Category
1108,beers,Beers
5092,energy_functional_drinks,Non-Alcoholic
5097,gourmet_food,Others
5340,grocery_food,Others
5411,household_kitchen,Others
8160,miscellaneous_retail,Others
8891,non-alcoholic,Non-Alcoholic
14292,operational_expenses,Others
14297,personal_care,Others
14382,promotional_items,Others


# Subcategory Engineering – Controlled Compression

## Objective
Redesign the Subcategory layer of the POS system by:

- Normalizing inconsistent naming
- Removing semantic duplicates
- Collapsing "ALL X" patterns
- Creating a governed subcategory framework
- Preserving product relationships

This ensures scalability and prevents future hierarchy explosion.

In [12]:
## Step 1: Normalize Raw Subcategory Values
#We standardize casing and remove whitespace inconsistencies.

df['Subcategory'] = df['Subcategory'].str.strip().str.upper()
df['Subcategory'].nunique()

921

In [13]:
## Step 2: Measure Subcategory Spread Per POS Parent
#This helps identify where compression is most needed.

sub_by_parent = df.groupby('POS_Parent_Category')['Subcategory'].nunique().reset_index()
sub_by_parent.columns = ['POS_Parent_Category', 'Unique_Subcategories']

sub_by_parent

,POS_Parent_Category,Unique_Subcategories
0,Beers,94
1,Non-Alcoholic,158
2,Others,524
3,Spirits,249
4,Tobacco,42
5,Wines,87


# Subcategory Re-Architecture – Controlled Taxonomy Design

The existing 921 subcategories are not true structural types.
They are semantic variations, store-level noise, and legacy labels.

We will now:

1. Design a controlled subcategory list per POS Parent.
2. Map products into this structure using rule-based classification.
3. Replace Clean_Subcategory with POS_Subcategory.

In [14]:
POS_SUBCATEGORY_MAP = {

    "Spirits": [
        "Whiskey",
        "Vodka",
        "Gin",
        "Rum",
        "Brandy",
        "Tequila",
        "Liqueur",
        "Bitters",
        "Other Spirits"
    ],

    "Wines": [
        "Red Wine",
        "White Wine",
        "Rose",
        "Sparkling Wine",
        "Fortified Wine",
        "Other Wine"
    ],

    "Beers": [
        "Lager",
        "Ale",
        "Stout",
        "Cider",
        "Other Beer"
    ],

    "Non-Alcoholic": [
        "Soft Drink",
        "Water",
        "Energy Drink",
        "Juice",
        "Mixer",
        "Other Non-Alcoholic"
    ],

    "Tobacco": [
        "Cigarettes",
        "Cigars",
        "Shisha",
        "Vape",
        "Other Tobacco"
    ],

    "Others": [
        "Snacks",
        "Household",
        "Accessories",
        "Misc Retail"
    ]
}

In [16]:
def map_pos_subcategory(row):
    parent = row['POS_Parent_Category']
    text = (str(row['Subcategory']) + " " + str(row['Product Label'])).upper()

    # SPIRITS
    if parent == "Spirits":
        if "WHISK" in text:
            return "Whiskey"
        elif "VODKA" in text:
            return "Vodka"
        elif "GIN" in text:
            return "Gin"
        elif "RUM" in text:
            return "Rum"
        elif "BRANDY" in text or "COGNAC" in text:
            return "Brandy"
        elif "TEQUILA" in text:
            return "Tequila"
        elif "LIQUEUR" in text:
            return "Liqueur"
        elif "BITTER" in text:
            return "Bitters"
        else:
            return "Other Spirits"

    # WINES
    elif parent == "Wines":
        if "RED" in text:
            return "Red Wine"
        elif "WHITE" in text:
            return "White Wine"
        elif "ROSE" in text:
            return "Rose"
        elif "SPARKLING" in text or "CHAMPAGNE" in text:
            return "Sparkling Wine"
        elif "PORT" in text or "FORTIFIED" in text:
            return "Fortified Wine"
        else:
            return "Other Wine"

    # BEERS
    elif parent == "Beers":
        if "STOUT" in text:
            return "Stout"
        elif "CIDER" in text:
            return "Cider"
        elif "ALE" in text:
            return "Ale"
        else:
            return "Lager"

    # NON-ALCOHOLIC
    elif parent == "Non-Alcoholic":
        if "WATER" in text:
            return "Water"
        elif "ENERGY" in text:
            return "Energy Drink"
        elif "JUICE" in text:
            return "Juice"
        elif "TONIC" in text or "SODA" in text:
            return "Mixer"
        else:
            return "Soft Drink"

    # TOBACCO
    elif parent == "Tobacco":
        if "CIGAR" in text:
            return "Cigars"
        elif "SHISHA" in text:
            return "Shisha"
        elif "VAPE" in text:
            return "Vape"
        else:
            return "Cigarettes"

    # OTHERS
    else:
        if "SNACK" in text:
            return "Snacks"
        elif "GLASS" in text or "OPENER" in text:
            return "Accessories"
        elif "HOUSE" in text or "KITCHEN" in text:
            return "Household"
        else:
            return "Misc Retail"

df['POS_Subcategory'] = df.apply(map_pos_subcategory, axis=1)

In [17]:
#Check new subcategories
df.groupby('POS_Parent_Category')['POS_Subcategory'].nunique()

,POS_Subcategory
POS_Parent_Category,
Beers,4
Non-Alcoholic,5
Others,4
Spirits,9
Tobacco,3
Wines,6


## Identify Oversized Subcategories

Large subcategories like "Other Spirits" or "Misc Retail" may indicate unclassified items.
We will extract them for review.

In [19]:
df.groupby(['POS_Parent_Category', 'POS_Subcategory']).size().reset_index(name='Count').sort_values(['POS_Parent_Category', 'Count'], ascending=[True, False])

,POS_Parent_Category,POS_Subcategory,Count
2,Beers,Lager,3309
1,Beers,Cider,554
0,Beers,Ale,82
3,Beers,Stout,39
7,Non-Alcoholic,Soft Drink,2386
5,Non-Alcoholic,Juice,1016
6,Non-Alcoholic,Mixer,781
8,Non-Alcoholic,Water,749
4,Non-Alcoholic,Energy Drink,474
11,Others,Misc Retail,6701
